## Rossmann - Data Cleaning & QA

This notebook checks and cleans the Rossmann datasets before SQL analysis, Power BI and modelling.

**1. Load data**

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("..") / "data" / "raw"
PROCESSED_PATH = Path("..") / "data" / "processed"

store_df = pd.read_csv(DATA_PATH / "store.csv")
train_df = pd.read_csv(DATA_PATH / "train.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")


C:\Users\Anaisa\AppData\Local\Temp\ipykernel_4208\896519782.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(DATA_PATH / "train.csv")


**2. Initial data inspection**

In [2]:
def inspect_dataset(name, df):
    print(f"{name.upper()} | shape: {df.shape}")
    display(df.head())
    df.info()
    print()

inspect_dataset("store", store_df)
inspect_dataset("train", train_df)
inspect_dataset("test", test_df)


STORE | shape: (1115, 10)


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   object 
 2   Assortment                 1115 non-null   object 
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    object 
dtypes: float64(5), int64(2), object(3)
memory usage: 87.2+ KB

TRAIN | shape: (1017209, 9)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  object
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 69.8+ MB

TEST | shape: (41088, 8)


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             41088 non-null  int64  
 1   Store          41088 non-null  int64  
 2   DayOfWeek      41088 non-null  int64  
 3   Date           41088 non-null  object 
 4   Open           41077 non-null  float64
 5   Promo          41088 non-null  int64  
 6   StateHoliday   41088 non-null  object 
 7   SchoolHoliday  41088 non-null  int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 2.5+ MB



**3. Column standardization**

In [3]:
# Review original column names before standardization
for name, df in {"store": store_df, "train": train_df, "test": test_df}.items():
    print(f"{name.capitalize()} columns: {df.columns.tolist()}")


Store columns: ['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']
Train columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
Test columns: ['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


In [4]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.lower()
    )
    return df

store_df = normalize_columns(store_df)
train_df = normalize_columns(train_df)
test_df = normalize_columns(test_df)

for name, df in {"store": store_df, "train": train_df, "test": test_df}.items():
    print(f"{name.capitalize()} columns: {df.columns.tolist()}")


Store columns: ['store', 'store_type', 'assortment', 'competition_distance', 'competition_open_since_month', 'competition_open_since_year', 'promo2', 'promo2_since_week', 'promo2_since_year', 'promo_interval']
Train columns: ['store', 'day_of_week', 'date', 'sales', 'customers', 'open', 'promo', 'state_holiday', 'school_holiday']
Test columns: ['id', 'store', 'day_of_week', 'date', 'open', 'promo', 'state_holiday', 'school_holiday']


**4. Data type validation**

In [5]:
# Convert calendar dates to datetime
train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"] = pd.to_datetime(test_df["date"])

# Use nullable integers for discrete calendar components that contain missing values
calendar_int_columns = [
    "competition_open_since_month",
    "competition_open_since_year",
    "promo2_since_week",
    "promo2_since_year",
]
store_df[calendar_int_columns] = store_df[calendar_int_columns].astype("Int64")

print("Train date dtype:", train_df["date"].dtype)
print("Test date dtype:", test_df["date"].dtype)
print(store_df[calendar_int_columns].dtypes)


Train date dtype: datetime64[ns]
Test date dtype: datetime64[ns]
competition_open_since_month    Int64
competition_open_since_year     Int64
promo2_since_week               Int64
promo2_since_year               Int64
dtype: object


**Note:** The month, year and week columns were changed from `float` to nullable integers (`Int64`) because they represent whole-number calendar values and contain missing values. `date` was converted to `datetime`.

In [6]:
# Inspect and standardize the mixed-type state_holiday column
print("Before standardization")
print("Train:", train_df["state_holiday"].unique())
print("Test:", test_df["state_holiday"].unique())

train_df["state_holiday"] = train_df["state_holiday"].astype("string")
test_df["state_holiday"] = test_df["state_holiday"].astype("string")

print("\nAfter standardization")
print("Train:", train_df["state_holiday"].unique())
print("Test:", test_df["state_holiday"].unique())


Before standardization
Train: ['0' 'a' 'b' 'c' 0]
Test: ['0' 'a']

After standardization
Train: <StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: string
Test: <StringArray>
['0', 'a']
Length: 2, dtype: string


**Note:** `state_holiday` had mixed types in the train data because `0` appeared as both an integer and a string. I converted the column to string in both train and test so the values are consistent.

**5. Numerical profile**

In [7]:
print("Store numerical profile")
display(store_df.describe().round(2))

print("Train sales/customer profile")
display(train_df[["sales", "customers"]].describe().round(2))


Store numerical profile


,store,competition_distance,competition_open_since_month,competition_open_since_year,promo2,promo2_since_week,promo2_since_year
count,1115.00,1112.00,761.0,761.0,1115.00,571.0,571.0
mean,558.00,5404.90,7.22,2008.67,0.51,23.6,2011.76
std,322.02,7663.17,3.21,6.2,0.50,14.14,1.67
min,1.00,20.00,1.0,1900.0,0.00,1.0,2009.0
25%,279.50,717.50,4.0,2006.0,0.00,13.0,2011.0
50%,558.00,2325.00,8.0,2010.0,1.00,22.0,2012.0
75%,836.50,6882.50,10.0,2013.0,1.00,37.0,2013.0
max,1115.00,75860.00,12.0,2015.0,1.00,50.0,2015.0


Train sales/customer profile


,sales,customers
count,1017209.00,1017209.00
mean,5773.82,633.15
std,3849.93,464.41
min,0.00,0.00
25%,3727.00,405.00
50%,5744.00,609.00
75%,7856.00,837.00
max,41551.00,7388.00


**6. Missing values**

In [8]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    })
    return summary[summary["missing_count"] > 0]

print("Store missing values")
display(missing_summary(store_df))

print("Train missing values")
display(missing_summary(train_df))

print("Test missing values")
display(missing_summary(test_df))


Store missing values


,missing_count,missing_pct
competition_distance,3,0.27
competition_open_since_month,354,31.75
competition_open_since_year,354,31.75
promo2_since_week,544,48.79
promo2_since_year,544,48.79
promo_interval,544,48.79


Train missing values


,missing_count,missing_pct


Test missing values


,missing_count,missing_pct
open,11,0.03


**Promo2 missing values**

In [9]:
# Check whether Promo2-related nulls are structural
promo2_missing_pattern = store_df.groupby("promo2")[[
    "promo2_since_week",
    "promo2_since_year",
    "promo_interval",
]].count()

display(promo2_missing_pattern)


,promo2_since_week,promo2_since_year,promo_interval
promo2,,,
0,0,0,0
1,571,571,571


**Note:** The nulls in `promo2_since_week`, `promo2_since_year` and `promo_interval` are expected when `promo2 = 0`, so they were kept as null rather than replaced with zero.

**Missing `open` values in test**

In [10]:
# Inspect rows with missing open values
missing_open = test_df[test_df["open"].isna()]
display(missing_open)

# Review Store 622's historical opening pattern by day of week
opening_pattern_622 = (
    train_df.loc[train_df["store"] == 622]
    .groupby("day_of_week")["open"]
    .value_counts()
    .unstack(fill_value=0)
)
display(opening_pattern_622)

# Investigate non-Sunday closures for Store 622
non_sunday_closures_622 = train_df.loc[
    (train_df["store"] == 622)
    & (train_df["open"] == 0)
    & (train_df["day_of_week"] != 7),
    ["date", "day_of_week", "promo", "state_holiday", "school_holiday"],
].sort_values("date")

display(non_sunday_closures_622)


,id,store,day_of_week,date,open,promo,state_holiday,school_holiday
479,480,622,4,2015-09-17,NaN,1,0,0
1335,1336,622,3,2015-09-16,NaN,1,0,0
2191,2192,622,2,2015-09-15,NaN,1,0,0
3047,3048,622,1,2015-09-14,NaN,1,0,0
4759,4760,622,6,2015-09-12,NaN,0,0,0
5615,5616,622,5,2015-09-11,NaN,0,0,0
6471,6472,622,4,2015-09-10,NaN,0,0,0
7327,7328,622,3,2015-09-09,NaN,0,0,0
8183,8184,622,2,2015-09-08,NaN,0,0,0
9039,9040,622,1,2015-09-07,NaN,0,0,0


open,0,1
day_of_week,,
1,6,128
2,1,134
3,3,132
4,8,127
5,6,129
6,0,134
7,134,0


,date,day_of_week,promo,state_holiday,school_holiday
1016716,2013-01-01,2,0,a,1
919711,2013-03-29,5,1,b,1
916366,2013-04-01,1,0,b,1
882916,2013-05-01,3,1,a,0
873996,2013-05-09,4,0,a,0
861731,2013-05-20,1,0,a,0
710091,2013-10-03,4,0,a,0
617546,2013-12-25,3,0,c,1
616431,2013-12-26,4,0,c,1
609741,2014-01-01,3,0,a,1


**Note:** All 11 missing `open` values belong to Store 622. In the train data, the store was closed on non-Sundays only on state holidays. Since the missing test dates are not state holidays, I filled these values with `1` (open).

In [11]:
# Impute missing open values and restore the binary integer type
test_df["open"] = test_df["open"].fillna(1).astype("int64")

print("Remaining missing open values:", test_df["open"].isna().sum())
print("open dtype:", test_df["open"].dtype)


Remaining missing open values: 0
open dtype: int64


**Competition missing values**

In [12]:
# Inspect stores with no competition distance
display(store_df.loc[store_df["competition_distance"].isna()])

competition_missing = (
    store_df["competition_open_since_month"].isna()
    & store_df["competition_open_since_year"].isna()
)

competition_date_mismatch = (
    store_df["competition_open_since_month"].isna()
    != store_df["competition_open_since_year"].isna()
)

print("Missing competition opening dates:", competition_missing.sum())
print("Month/year mismatches:", competition_date_mismatch.sum())
print(
    "Known competition distance among these stores:",
    store_df.loc[competition_missing, "competition_distance"].notna().sum(),
)
print(
    "Missing competition distance among these stores:",
    store_df.loc[competition_missing, "competition_distance"].isna().sum(),
)


,store,store_type,assortment,competition_distance,competition_open_since_month,competition_open_since_year,promo2,promo2_since_week,promo2_since_year,promo_interval
290,291,d,a,NaN,<NA>,<NA>,0,<NA>,<NA>,NaN
621,622,a,c,NaN,<NA>,<NA>,0,<NA>,<NA>,NaN
878,879,d,a,NaN,<NA>,<NA>,1,5,2013,"Feb,May,Aug,Nov"


Missing competition opening dates: 354
Month/year mismatches: 0
Known competition distance among these stores: 351
Missing competition distance among these stores: 3


**Note:** `competition_open_since_month` and `competition_open_since_year` are missing together for 354 stores. For 351 of them, `competition_distance` is available, so the competitor is known but the opening date is not. Three stores have no competition information.

I kept these values as null because filling them would mean creating information that is not available in the dataset.

**7. Duplicate check**

In [13]:
duplicate_summary = pd.Series({
    "store_duplicates": store_df.duplicated().sum(),
    "train_duplicates": train_df.duplicated().sum(),
    "test_duplicates": test_df.duplicated().sum(),
    "store_key_is_unique": store_df["store"].is_unique,
})

display(duplicate_summary.to_frame("result"))


,result
store_duplicates,0
train_duplicates,0
test_duplicates,0
store_key_is_unique,True


**8. Consistency and business rules**

In [14]:
# Review categorical domains
categorical_values = {
    "store_type": store_df["store_type"].unique().tolist(),
    "assortment": store_df["assortment"].unique().tolist(),
    "promo2": store_df["promo2"].unique().tolist(),
    "open": train_df["open"].unique().tolist(),
    "promo": train_df["promo"].unique().tolist(),
    "state_holiday": train_df["state_holiday"].unique().tolist(),
    "school_holiday": train_df["school_holiday"].unique().tolist(),
}

for column, values in categorical_values.items():
    print(f"{column}: {values}")


store_type: ['c', 'a', 'd', 'b']
assortment: ['a', 'c', 'b']
promo2: [0, 1]
open: [1, 0]
promo: [1, 0]
state_holiday: ['0', 'a', 'b', 'c']
school_holiday: [1, 0]


In [15]:
# Business-rule validation
business_checks = pd.Series({
    "day_of_week within 1-7": (
        train_df["day_of_week"].between(1, 7).all()
        and test_df["day_of_week"].between(1, 7).all()
    ),
    "competition month within 1-12": (
        store_df["competition_open_since_month"].dropna().between(1, 12).all()
    ),
    "promo2 week within 1-53": (
        store_df["promo2_since_week"].dropna().between(1, 53).all()
    ),
    "sales are non-negative": (train_df["sales"] >= 0).all(),
    "customers are non-negative": (train_df["customers"] >= 0).all(),
    "competition distance is non-negative": (
        store_df["competition_distance"].dropna() >= 0
    ).all(),
    "open is binary": train_df["open"].isin([0, 1]).all() and test_df["open"].isin([0, 1]).all(),
    "promo is binary": train_df["promo"].isin([0, 1]).all() and test_df["promo"].isin([0, 1]).all(),
    "school_holiday is binary": (
        train_df["school_holiday"].isin([0, 1]).all()
        and test_df["school_holiday"].isin([0, 1]).all()
    ),
    "promo2 is binary": store_df["promo2"].isin([0, 1]).all(),
    "state_holiday categories are valid": (
        set(train_df["state_holiday"].dropna().unique()).issubset({"0", "a", "b", "c"})
        and set(test_df["state_holiday"].dropna().unique()).issubset({"0", "a", "b", "c"})
    ),
    "closed stores have zero sales/customers": not (
        (train_df["open"] == 0)
        & ((train_df["sales"] != 0) | (train_df["customers"] != 0))
    ).any(),
    "promo2 inactive implies promo2 details missing": store_df.loc[
        store_df["promo2"] == 0,
        ["promo2_since_week", "promo2_since_year", "promo_interval"],
    ].isna().all().all(),
    "store key is unique": store_df["store"].is_unique,
}, name="passed")

display(business_checks.to_frame())


,passed
day_of_week within 1-7,True
competition month within 1-12,True
promo2 week within 1-53,True
sales are non-negative,True
customers are non-negative,True
competition distance is non-negative,True
open is binary,True
promo is binary,True
school_holiday is binary,True
promo2 is binary,True


**9. Final validation**

In [16]:
# Required fields should be complete after treatment
required_store_columns = ["store", "store_type", "assortment", "promo2"]

assert store_df[required_store_columns].notna().all().all()
assert train_df.notna().all().all()
assert test_df.notna().all().all()

# Structural integrity checks
assert store_df.duplicated().sum() == 0
assert train_df.duplicated().sum() == 0
assert test_df.duplicated().sum() == 0
assert store_df["store"].is_unique
assert business_checks.all()

print("Final data quality validation passed.")
print("\nExpected remaining nulls in store_df:")
display(missing_summary(store_df))


Final data quality validation passed.

Expected remaining nulls in store_df:


,missing_count,missing_pct
competition_distance,3,0.27
competition_open_since_month,354,31.75
competition_open_since_year,354,31.75
promo2_since_week,544,48.79
promo2_since_year,544,48.79
promo_interval,544,48.79


**10. Merge datasets**

In [17]:
train_shape_before = train_df.shape
test_shape_before = test_df.shape

train_merged = train_df.merge(
    store_df,
    on="store",
    how="left",
    validate="many_to_one",
)

test_merged = test_df.merge(
    store_df,
    on="store",
    how="left",
    validate="many_to_one",
)


**11. Validate merge**

In [18]:
# Row counts must be preserved and every store must match store_df
assert len(train_merged) == len(train_df)
assert len(test_merged) == len(test_df)
assert train_merged["store_type"].notna().all()
assert test_merged["store_type"].notna().all()

print("Train before merge:", train_shape_before)
print("Train after merge:", train_merged.shape)
print("Test before merge:", test_shape_before)
print("Test after merge:", test_merged.shape)
print("Merge validation passed.")


Train before merge: (1017209, 9)
Train after merge: (1017209, 18)
Test before merge: (41088, 8)
Test after merge: (41088, 17)
Merge validation passed.


**Note:** `store_df` has one row per store. `validate="many_to_one"` is used in the merge to make sure each train/test row matches only one store.

**12. Export processed data**

In [19]:
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Preserve cleaned source tables for SQL / relational analysis
store_df.to_csv(PROCESSED_PATH / "store_clean.csv", index=False)
train_df.to_csv(PROCESSED_PATH / "train_clean.csv", index=False)
test_df.to_csv(PROCESSED_PATH / "test_clean.csv", index=False)

# Export enriched datasets for EDA / BI / modelling
train_merged.to_csv(PROCESSED_PATH / "train_enriched.csv", index=False)
test_merged.to_csv(PROCESSED_PATH / "test_enriched.csv", index=False)

print(
    "Export completed successfully. Clean source tables and enriched datasets "
    "were saved to data/processed."
)


Export completed successfully. Clean source tables and enriched datasets were saved to data/processed.
